# Controlled pretraining ablations (R2 2.M5)

Reads tables from `../data/` and writes figures to `../output/`.
Source analysis: `../../NOTEBOOK_PROVENANCE.tsv`.

Reviewer 2.M5 asked for controlled pretraining ablations of the fragment-specific design. A full LEAF-1 model and
four variants were pretrained from scratch on the same Cristiano-free corpus (30,000 updates, effective batch 768,
seed 42; the full and flank-free models also with seed 43): no [CUT] tokens, [CUT] masked at the base rate (0.15
instead of 0.30), no flanking sequence, and a constant placeholder for the modality token. Every model was applied to
the same 10,000 fragments per sample as Figure 4.

- **a**: inputs of the five models (schematic; no data).
- **b**: ΔAUC vs the full model, full feature set; linear SVM (Figure 4 protocol, 14 tasks) and MIL (15 tasks, mean
  inner-fold test AUROC). References as in the response draft: linear SVM single-seed variants vs the seed-42 full
  model; MIL single-seed variants vs the full model averaged over both seeds; flank-free seed-averaged on both sides.
- **c**: the same comparison for single readouts (linear SVM): [CUT] readout and body readout.

Paired over tasks: mean Δ, BCa bootstrap 95% CI (10,000 resamples), number of tasks lower.

Plotting code is copied from `src/cfdna/ablations_2m5/make_ablations.py`.

Output: `cfdna_pretraining_ablations` (assembled a-c, 132 mm wide).


In [1]:
from pathlib import Path
import sys

# Self-contained: works when run from notebook/, from the modality folder, or from the repo root.
nb_dir = Path.cwd()
if (nb_dir / "data").exists() and (nb_dir / "output").exists():
    MOD_DIR = nb_dir
elif (nb_dir.parent / "data").exists():
    MOD_DIR = nb_dir.parent
else:
    MOD_DIR = nb_dir
DATA = MOD_DIR / "data"
OUT = MOD_DIR / "output"
OUT.mkdir(exist_ok=True)

# Locate the shared style helper (src/leaf1figures) by walking up.
_p = nb_dir
while _p != _p.parent and not (_p / "src" / "leaf1figures").exists():
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

from leaf1figures.style import OKABE_ITO, savefig, set_paper_style
set_paper_style()
print("DATA:", DATA.relative_to(_p))
print("OUT :", OUT.relative_to(_p))

DATA: cfdna/data
OUT : cfdna/output


In [2]:
# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

plt.rcParams.update({'lines.linewidth': 1, 'xtick.labelsize': 6, 'ytick.labelsize': 6})

FIG = pd.read_csv(DATA / "cfdna_pretraining_ablations_deltas.csv", float_precision="round_trip")
assert len(FIG) == 10 and set(FIG.panel) == {'b', 'c'}

COL = {'probe': OKABE_ITO['blue'], 'mil': OKABE_ITO['vermillion']}
CLF_SHORT = {'probe': 'Linear SVM (14 tasks)', 'mil': 'MIL (15 tasks)'}

# panel b: the four ablations, full feature set, both classifiers
B = FIG[FIG.panel == 'b'].copy()
order_b = ['No [CUT] tokens', 'Equal [CUT] masking (0.15)', 'No flanking sequence (2 seeds)', 'No modality token']
B['o'] = B.label.map({l: i for i, l in enumerate(order_b)}); B = B.sort_values(['o', 'classifier'])

# panel c: single readouts (linear SVM), model − full model
CROWS = [('[CUT] readout', None), ('Equal [CUT] masking (0.15)', 'cut_eqmask'),
         ('Body readout', None), ('No [CUT] tokens', 'body_nocut'), ('Equal [CUT] masking (0.15)', 'body_eqmask')]
Cc = FIG.set_index('id').loc[[i for _, i in CROWS if i]].reset_index()
FIGROWS = pd.concat([B.drop(columns='o'), Cc], ignore_index=True)
print(FIGROWS[['panel', 'label', 'classifier', 'readout', 'delta', 'ci_low', 'ci_high', 'n_lower', 'n_tasks']]
      .round(3).to_string(index=False))


panel                          label classifier readout  delta  ci_low  ci_high  n_lower  n_tasks
    b                No [CUT] tokens        mil    full -0.088  -0.119   -0.063       15       15
    b                No [CUT] tokens      probe    full -0.087  -0.116   -0.058       13       14
    b     Equal [CUT] masking (0.15)        mil    full -0.019  -0.032   -0.007       11       15
    b     Equal [CUT] masking (0.15)      probe    full -0.018  -0.045    0.003        8       14
    b No flanking sequence (2 seeds)        mil    full  0.020   0.001    0.040        6       15
    b No flanking sequence (2 seeds)      probe    full  0.058   0.030    0.083        3       14
    b              No modality token      probe    full  0.009  -0.007    0.029        5       14
    c     Equal [CUT] masking (0.15)      probe     cut -0.035  -0.069   -0.005        9       14
    c                No [CUT] tokens      probe    body -0.021  -0.039   -0.007       11       14
    c     Equal [CUT

In [3]:
# ── Panel drawing (verbatim from make_ablations.py) ───────────────────────────
# panel a data
SEG = {'mod': ('#CC79A7', 'MOD'), 'ph': ('#BBBBBB', '·'), 'cut': ('#000000', ''), 'flank': ('#D9E8F5', 'flank'),
       'body': ('#E69F00', 'body'), 'joint': ('#D9E8F5', '')}
MODELS = [
    ('Full LEAF-1', [('mod', .7), ('flank', 3), ('cut', .25), ('body', 1.5), ('cut', .25), ('flank', 3)], ('2,048', '0.30')),
    ('No [CUT] tokens', [('mod', .7), ('joint', 8)], ('2,048', '–')),
    ('Equal [CUT] masking', [('mod', .7), ('flank', 3), ('cut', .25), ('body', 1.5), ('cut', .25), ('flank', 3)], ('2,048', '0.15')),
    ('No flanking sequence', [('mod', .7), ('cut', .25), ('body', 1.5), ('cut', .25)], ('512', '0.30')),
    ('No modality token', [('ph', .7), ('flank', 3), ('cut', .25), ('body', 1.5), ('cut', .25), ('flank', 3)], ('2,048', '0.30')),
]


XMAX_A = 12.0  # right edge of panel a = right edge of the notes column (aligned with panel c in the assembly)


def draw_a(ax):
    for i, (name, segs, note) in enumerate(MODELS):
        x = 0
        for kind, w in segs:
            c, lab = SEG[kind]
            ax.add_patch(Rectangle((x, -i - 0.32), w, 0.64, facecolor=c, edgecolor='white', lw=0.6))
            if kind == 'joint':  # body inside one continuous string, no boundary marks
                ax.add_patch(Rectangle((x + 3.25, -i - 0.32), 1.5, 0.64, facecolor='#F5D9A0', edgecolor='none'))
                ax.text(x + 4.0, -i, 'flank + body + flank, one string', ha='center', va='center', fontsize=5.5)
            elif lab:
                ax.text(x + w / 2, -i, lab, ha='center', va='center', fontsize=5.5,
                        color='white' if kind == 'mod' else 'black')
            x += w
        ax.text(-0.25, -i, name, ha='right', va='center', fontsize=6.5)
        ax.text(XMAX_A - 1.5, -i, note[0], ha='right', va='center', fontsize=6)
        ax.text(XMAX_A - 0.1, -i, note[1], ha='right', va='center', fontsize=6)
    ax.text(XMAX_A - 1.5, 0.45, 'Tokens', ha='right', va='bottom', fontsize=6, fontweight='bold')
    ax.text(XMAX_A - 0.1, 0.45, '[CUT]\nmask', ha='right', va='bottom', fontsize=6, fontweight='bold', linespacing=1.0)
    ax.add_patch(Rectangle((0.2, -len(MODELS) - 0.15), 0.25, 0.3, facecolor='black'))
    ax.text(0.55, -len(MODELS), '[CUT] token', va='center', fontsize=5.5)
    ax.set_xlim(-0.2, XMAX_A); ax.set_ylim(-len(MODELS) - 0.4, 1.3); ax.axis('off')


def draw_b(ax, legend_y=-0.27, TXT_X=1.22, N_X=1.55):
    for k in range(len(order_b)):
        if k % 2 == 0:
            ax.axhspan(k - 0.5, k + 0.5, color='#F3F3F3', zorder=0, lw=0)
    for xv in [-0.1, -0.05, 0.05, 0.1]:
        ax.axvline(xv, color='#E6E6E6', lw=0.5, zorder=1)
    ax.axvline(0, color='#555555', lw=0.75, zorder=1)
    for r in B.itertuples():
        yy = r.o + {'probe': -0.19, 'mil': 0.19}[r.classifier]
        ax.hlines(yy, r.ci_low, r.ci_high, color=COL[r.classifier], lw=1.2, zorder=2)
        ax.scatter(r.delta, yy, s=16, color=COL[r.classifier], edgecolor='white', lw=0.5, zorder=3)
        ax.text(TXT_X, yy, f'{r.delta:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]'.replace('-', '−'),
                transform=ax.get_yaxis_transform(), ha='center', va='center', fontsize=5.5, color=COL[r.classifier])
        ax.text(N_X, yy, f'{r.n_lower}/{r.n_tasks}', transform=ax.get_yaxis_transform(), ha='center', va='center',
                fontsize=5.5, color=COL[r.classifier])
    for x_, t in [(TXT_X, 'ΔAUC [95% CI]'), (N_X, 'Tasks\nlower')]:
        ax.text(x_, -0.58, t, transform=ax.get_yaxis_transform(), ha='center', va='bottom', fontsize=5.5,
                fontweight='bold', linespacing=1.0)
    ax.set_yticks(range(len(order_b))); ax.set_yticklabels(order_b); ax.tick_params(axis='y', length=0)
    ax.set_ylim(len(order_b) - 0.5, -0.5); ax.set_xlim(-0.13, 0.11)
    ax.set_xticks([-0.1, -0.05, 0, 0.05, 0.1]); ax.set_xticklabels(['−0.10', '−0.05', '0', '+0.05', '+0.10'])
    ax.set_xlabel('ΔAUC vs full LEAF-1 model'); ax.spines['left'].set_visible(False)
    h = [plt.Line2D([], [], marker='o', ls='-', color=COL[c], ms=3.5, lw=1.2, label=CLF_SHORT[c]) for c in ['probe', 'mil']]
    ax.legend(handles=h, loc='upper center', bbox_to_anchor=(0.5, legend_y), ncol=2, frameon=False, handlelength=1.5)


def draw_c(ax, TXT_X=1.22, N_X=1.55):
    """Same format as panel b: ΔAUC vs the full model for single readouts, linear SVM only."""
    ydata = [k for k, (_, i) in enumerate(CROWS) if i]
    for j, k in enumerate(ydata):
        if j % 2 == 0:
            ax.axhspan(k - 0.5, k + 0.5, color='#F3F3F3', zorder=0, lw=0)
    for xv in [-0.1, -0.05, 0.05, 0.1]:
        ax.axvline(xv, color='#E6E6E6', lw=0.5, zorder=1)
    ax.axvline(0, color='#555555', lw=0.75, zorder=1)
    for k, (lab, i) in enumerate(CROWS):
        if i is None:  # group header
            ax.text(-0.02, k, lab, transform=ax.get_yaxis_transform(), ha='right', va='center', fontsize=6,
                    fontweight='bold')
            continue
        r = Cc.set_index('id').loc[i]
        ax.hlines(k, r.ci_low, r.ci_high, color=COL['probe'], lw=1.2, zorder=2)
        ax.scatter(r.delta, k, s=16, color=COL['probe'], edgecolor='white', lw=0.5, zorder=3)
        ax.text(TXT_X, k, f'{r.delta:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}]'.replace('-', '−'),
                transform=ax.get_yaxis_transform(), ha='center', va='center', fontsize=5.5, color=COL['probe'])
        ax.text(N_X, k, f'{r.n_lower}/{r.n_tasks}', transform=ax.get_yaxis_transform(), ha='center', va='center',
                fontsize=5.5, color=COL['probe'])
    for x_, t in [(TXT_X, 'ΔAUC [95% CI]'), (N_X, 'Tasks\nlower')]:
        ax.text(x_, -0.58, t, transform=ax.get_yaxis_transform(), ha='center', va='bottom', fontsize=5.5,
                fontweight='bold', linespacing=1.0)
    ax.set_yticks(ydata); ax.set_yticklabels([CROWS[k][0] for k in ydata]); ax.tick_params(axis='y', length=0)
    ax.set_ylim(len(CROWS) - 0.5, -0.5); ax.set_xlim(-0.13, 0.11)
    ax.set_xticks([-0.1, -0.05, 0, 0.05, 0.1]); ax.set_xticklabels(['−0.10', '−0.05', '0', '+0.05', '+0.10'])
    ax.set_xlabel('ΔAUC vs full LEAF-1 model, linear SVM'); ax.spines['left'].set_visible(False)


In [4]:
# ── Assembled figure (a-c) ───────────────────────────────────────────────────
# assembled supplementary figure, 5.2 in (132 mm) wide; axes placed in inches
W, H = 5.2, 5.3
fig = plt.figure(figsize=(W, H))


def axes_in(l, b, w, h):
    return fig.add_axes([l / W, b / H, w / W, h / H])


ax_a = axes_in(1.25, 4.0, 3.85, 1.25); draw_a(ax_a)
ax_b = axes_in(1.25, 2.3, 2.3, 1.3); draw_b(ax_b, legend_y=-0.28, TXT_X=1.255, N_X=1.585)
ax_c = axes_in(1.25, 0.35, 2.3, 1.0); draw_c(ax_c, TXT_X=1.255, N_X=1.585)
for lab, (x, y) in {'a': (0.02, 5.27), 'b': (0.02, 3.8), 'c': (0.02, 1.62)}.items():
    fig.text(x / W, y / H, lab, fontsize=9, fontweight='bold', va='top')
savefig(fig, OUT, 'cfdna_pretraining_ablations', data=FIGROWS); plt.close(fig)


saved: cfdna/output/cfdna_pretraining_ablations.pdf
saved: cfdna/output/cfdna_pretraining_ablations.png
saved: cfdna/output/cfdna_pretraining_ablations.svg
saved: cfdna/output/cfdna_pretraining_ablations.tsv
